# Avatar Video Studio on Google Colab
Runtime → Change runtime type → **T4 GPU**. Run the cells in order. Everything is open source and local to the Colab machine (no API keys).

⚠️ This notebook is **untested**; the Ollama install cell in particular.

In [ ]:
!nvidia-smi | head -5

## 1) Upload `avatar-video-studio.zip`

In [ ]:
from google.colab import files
files.upload()
!unzip -qo avatar-video-studio.zip
%cd avatar-video-studio

## 2) Install (libraries, Chromium, MuseTalk environment)

In [ ]:
!python studio.py setup --skip-weights --skip-llm

## 3) MuseTalk weights: restore from your Drive, or download once and save them

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
W = '/content/drive/MyDrive/avatar_studio_models'
if os.path.isdir(W):
    !cp -r $W third_party/MuseTalk/models
    print('restored from Drive')
else:
    !.venv-musetalk/bin/python download_weights.py
    !cp -r third_party/MuseTalk/models $W
    print('downloaded and saved to Drive:', W)

## 4) Local language model (Ollama + Qwen3, open source)

In [ ]:
import os, subprocess, time
os.environ['OLLAMA_HOST'] = '127.0.0.1:11500'
os.environ['OLLAMA_MODELS'] = '/content/avatar-video-studio/models_cache/ollama'
!curl -fsSL https://ollama.com/install.sh | sh
subprocess.Popen(['ollama', 'serve']); time.sleep(5)
!ollama pull qwen3:8b

## 5) Run the UI, then open the `gradio.live` link

In [ ]:
import os
os.environ['SHARE'] = '1'      # public link, needed on Colab
os.environ['LLM_MODEL'] = 'qwen3:8b'
!python studio.py run --no-ollama